# Bab 11. Analisis Data Eksploratif

Kode pendamping buku *Python untuk Machine Learning dan Data
Science*. Jalankan selnya berurutan dari atas, sebab sebagian
sel memakai peubah dari sel sebelumnya.

Notebook ini dibangkitkan dari naskah buku. Jangan disunting di
sini, sunting listing pada berkas `.tex` lalu bangkitkan ulang.

## Persiapan

Bab ini melanjutkan contoh dari bab sebelumnya. Jalankan sel ini lebih dahulu supaya datanya tersedia.

In [ ]:
import pandas as pd
import numpy as np
from siapkan import kue_tiga_bulan, kue_setahun

produk, trans, g = kue_tiga_bulan()
g["bulan"] = g["tanggal"].dt.to_period("M")

prod_th, df = kue_setahun()
df = df.merge(prod_th, on="id_produk", how="left")
df["omzet"] = df["jumlah"] * df["harga"]
df["bulan"] = df["tanggal"].dt.to_period("M")

## 1. Melihat cacah di balik total

In [ ]:
b = g[g["kategori"] == "basah"]
print(b.groupby("bulan")["omzet"]
       .agg(["count", "sum", "median"]))

Keluaran yang diharapkan:

```
         count      sum    median
bulan
2025-11      3   790000  180000.0
2025-12      2   750000  375000.0
2026-01      3  1700000  540000.0
```

## 2. Empat perintah pembuka

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.head(3))
print(df["jumlah"].describe())

Keluaran yang diharapkan:

```
(528, 4)

tanggal      datetime64[us]
id_produk               str
jumlah              float64
channel                 str

   tanggal id_produk  jumlah channel
2025-10-23        P5     4.0    toko
2025-11-07        P6     2.0    toko
2025-04-16        P2     9.0    toko

count    519.00
mean      11.63
std       22.60
min       -5.00
25%        5.00
50%        8.00
75%        12.00
max      251.00
```

## 3. Memeriksa cacat

In [ ]:
print(df.duplicated().sum())
print(df["jumlah"].isna().sum())
print((df["jumlah"] < 0).sum())
print(sorted(df["channel"].unique()))

Keluaran yang diharapkan:

```
5
9
3
['Toko', 'online', 'reseller', 'toko']
```

## 4. Membersihkan data

In [ ]:
d = df.drop_duplicates().copy()
d["channel"] = d["channel"].str.lower()
d = d[d["jumlah"].isna() | (d["jumlah"] > 0)]
d = d.dropna(subset=["jumlah"])
d["jumlah"] = d["jumlah"].astype("int64")

print(d.shape)
print(sorted(d["channel"].unique()))

Keluaran yang diharapkan:

```
(511, 4)
['online', 'reseller', 'toko']
```

## 5. Mengukur kemencengan

In [ ]:
o = d["omzet"]
print(f"{o.mean():,.0f}")
print(f"{o.median():,.0f}")
print(f"{o.skew():.2f}")
print(f"{np.log10(o).skew():.2f}")

Keluaran yang diharapkan:

```
936,646
600,000
9.74
0.34
```

## 6. Kaidah jarak antarkuartil

In [ ]:
q1, q3 = o.quantile([0.25, 0.75])
iqr = q3 - q1
atas = q3 + 1.5 * iqr

print(f"{q1:,.0f} {q3:,.0f} {iqr:,.0f} {atas:,.0f}")
print((o > atas).sum(), len(o))
print(f"{o[o > atas].sum() / o.sum():.1%}")

Keluaran yang diharapkan:

```
330,000 1,017,000 687,000 2,047,500
27 511
32.8%
```

## 7. Dua jenis korelasi

In [ ]:
k = d[["jumlah", "harga", "omzet"]]
print(k.corr().round(3))
print(k.corr(method="spearman").round(3))

Keluaran yang diharapkan:

```
        jumlah  harga  omzet
jumlah   1.000  0.028  0.903
harga    0.028  1.000  0.207
omzet    0.903  0.207  1.000

        jumlah  harga  omzet
jumlah   1.000  0.063  0.859
harga    0.063  1.000  0.518
omzet    0.859  0.518  1.000
```